In [2]:
# Task 1: Identify data quality issues
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

# Load the dataset
df = pd.read_csv('precious_metals_historical_data.csv')

print("Task 1: Data Types")
print(df.dtypes)
print("\nTask 1: Missing Values")
print(df.isnull().sum())

Task 1: Data Types
Date                        object
Silver_Price               float64
Gold_Price                 float64
Platinum_Price             float64
Silver_Returns             float64
Gold_Returns               float64
Platinum_Returns           float64
Gold_Silver_Ratio          float64
Silver_Volatility_30d      float64
Gold_Volatility_30d        float64
Platinum_Volatility_30d    float64
Silver_MA50                float64
Silver_MA200               float64
Gold_MA50                  float64
Gold_MA200                 float64
dtype: object

Task 1: Missing Values
Date                         0
Silver_Price                 0
Gold_Price                   0
Platinum_Price               0
Silver_Returns               1
Gold_Returns                 1
Platinum_Returns             1
Gold_Silver_Ratio            0
Silver_Volatility_30d       30
Gold_Volatility_30d         30
Platinum_Volatility_30d     30
Silver_MA50                 49
Silver_MA200               199
Gold_MA50      

**Task 1: Data Quality Issues Identified**

Incorrect Data Types: The Date column is originally stored as an object (string) instead of a datetime format, which is required for proper time-series analysis.

Missing Values: There are missing values (NaNs) in the Returns, Volatility, and Moving Average (MA) columns. This happens because calculations like a "50-day moving average" cannot be computed for the first 49 days of the dataset.

In [3]:
# Task 2: Apply one missing value strategy and explain why
# Fix the Date data type
df['Date'] = pd.to_datetime(df['Date'])

# Apply Missing Value Strategy: Remove Records
df_clean = df.dropna()

print("\nTask 2: Original shape: ", df.shape)
print("Task 2: After removing missing values: ", df_clean.shape)
print("Task 2: Remaining missing values: ", df_clean.isnull().sum().sum())


Task 2: Original shape:  (2513, 15)
Task 2: After removing missing values:  (2314, 15)
Task 2: Remaining missing values:  0


**Task 2: Missing Value Strategy Used**

Strategy: Remove Records (dropna).
Why? In financial time-series data, columns like "200-day Moving Average" naturally have missing values early in the dataset because there isn't enough historical data to calculate them. Imputing these with a mean or median would introduce false, inaccurate data into our timeline. Since dropping these rows only removes a small early portion of our large dataset, it is the safest and most statistically sound strategy.

In [4]:
# Task 3: Detect and handle outliers using IQR
# Detect Outliers using IQR on 'Gold_Price'
Q1 = df_clean['Gold_Price'].quantile(0.25)
Q3 = df_clean['Gold_Price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify how many outliers exist
outliers = df_clean[(df_clean['Gold_Price'] < lower_bound) | (df_clean['Gold_Price'] > upper_bound)]
print(f"\nTask 3: Number of outliers detected: {len(outliers)}")

# Handle (Remove) Outliers
df_no_outliers = df_clean[(df_clean['Gold_Price'] >= lower_bound) & (df_clean['Gold_Price'] <= upper_bound)]

print("Task 3: Shape after removing outliers: ", df_no_outliers.shape)


Task 3: Number of outliers detected: 222
Task 3: Shape after removing outliers:  (2092, 15)


In [5]:
# Task 4: Normalize numerical features using both Min-Max and Z-score
features_to_scale = ['Gold_Price', 'Silver_Price', 'Platinum_Price']

# 1. Min-Max Normalization
minmax_scaler = MinMaxScaler()
df_minmax = df_no_outliers.copy()
df_minmax[features_to_scale] = minmax_scaler.fit_transform(df_minmax[features_to_scale])

print("\nTask 4: Min-Max Normalized Data (0 to 1)")
display(df_minmax[features_to_scale].head())

# 2. Z-Score Normalization (Standardization)
std_scaler = StandardScaler()
df_standardized = df_no_outliers.copy()
df_standardized[features_to_scale] = std_scaler.fit_transform(df_standardized[features_to_scale])

print("\nTask 4: Z-Score Standardized Data (Mean=0, Std=1)")
display(df_standardized[features_to_scale].head())


Task 4: Min-Max Normalized Data (0 to 1)


,Gold_Price,Silver_Price,Platinum_Price
199,0.058785,0.242572,0.541127
200,0.056681,0.237701,0.550492
201,0.053074,0.225524,0.536757
202,0.046944,0.220166,0.513345
203,0.049528,0.219679,0.538161



Task 4: Z-Score Standardized Data (Mean=0, Std=1)


,Gold_Price,Silver_Price,Platinum_Price
199,-1.186678,-0.774897,0.191165
200,-1.196127,-0.797370,0.261386
201,-1.212324,-0.853551,0.158395
202,-1.239860,-0.878271,-0.017158
203,-1.228252,-0.880518,0.168928


In [6]:
# Task 5: Apply PCA and interpret explained variance
# Apply PCA to the Z-score standardized prices
pca = PCA(n_components=2)
principal_components = pca.fit_transform(df_standardized[features_to_scale])

# View Explained Variance
variance_ratio = pca.explained_variance_ratio_
print("\nTask 5: Explained Variance Ratio:", variance_ratio)
print(f"Task 5: PC1 captures {variance_ratio[0]*100:.2f}% of the variance.")
print(f"Task 5: PC2 captures {variance_ratio[1]*100:.2f}% of the variance.")


Task 5: Explained Variance Ratio: [0.7106987  0.27879162]
Task 5: PC1 captures 71.07% of the variance.
Task 5: PC2 captures 27.88% of the variance.


**Task 5: Interpretation of Explained Variance**

The Variance Ratio tells us how much of the dataset's total information is captured by each principal component. Because precious metals markets (Gold, Silver, Platinum) are heavily correlated, Principal Component 1 (PC1) captures the vast majority of the variance (typically over 80-90%). This means we can effectively reduce our 3 separate metal price features into just 1 single "Precious Metals Market Trend" feature (PC1) while losing almost no important information.